[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Gradient Descent by Hand


## What you will be able to do

Write a training loop by hand in PyTorch and in TensorFlow, choose a learning rate that converges,
and read from the loss why a loop that raised nothing failed to train.

## The idea

### The problem

The **Why PyTorch and TensorFlow** notebook fitted the January mean of the made-up stations with a
loop of a few lines and moved on. Those lines are the whole of training. Every optimizer in both
libraries runs them with its own rule for the step, and Keras's `fit` runs them for you. When
training goes wrong, the library usually raises nothing. The symptom shows in the loss, which sits
still, creeps down, jumps around or turns into `nan`, and reading it means knowing what each line of
the loop did.

This notebook writes the loop with nothing hidden, for a linear model of the January mean from the
four standardized features. A linear model has an exact best answer, which NumPy's least squares
computes directly, so every number the loop reaches can be checked.

### A formal definition

> **Gradient descent** repeats one update, in which each parameter moves against the derivative of
> the loss with respect to it, by the **learning rate** times that derivative. **Full-batch** gradient
> descent computes the loss on every training example at each step. **Mini-batch** gradient descent
> computes it on a small random sample, a **batch**, and an **epoch** is one pass through all the
> training examples, batch by batch.

### Why it works that way

The derivative says which way the loss rises and how steeply, so a small step against it lowers the
loss. The step has to be small enough that the slope has barely changed along the way. A learning
rate that is too small makes training creep. One that is too large makes each step land farther up
the other side of the valley than it started, so the loss grows with every step until it is
infinite.

What counts as small enough depends on the data. Elevation is measured in meters, with values in
the hundreds, so its gradient is far larger than latitude's, and one learning rate cannot suit
both. Standardizing every feature to a standard deviation of 1 gives them gradients of similar
size. Two features that rise together, as latitude and longitude do across Norway, leave the loss
almost flat when one weight rises and the other falls by the same amount, so gradient descent moves
slowly in that direction even after standardizing.

A mini-batch gives an estimate of the gradient from a fraction of the data. The estimate is noisy,
but it costs a fraction as much, so one pass through the data takes many steps instead of one.

### Where this shows up

The **Losses and Optimizers** notebook hands the step to `torch.optim` and to Keras's optimizers,
the **Batches and Data Pipelines** notebook hands the batches to `DataLoader` and `tf.data`, and the
**Training Loops** notebook adds a validation pass. Each is this loop with one part given to the
library.

### What this notebook covers

- one step of the loop, part by part
- the whole loop, checked against NumPy's least squares answer
- learning rates side by side
- two features that move together, and why their weights settle slowly
- why the features are standardized
- mini-batches and epochs
- the same loop in TensorFlow
- two failures that raise nothing until the loss shows them, and a mistake that raises a `TypeError`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch

w = torch.tensor(0.0, requires_grad=True)
for step in range(4):
    loss = (w - 3.0) ** 2
    loss.backward()
    with torch.no_grad():
        w -= 0.25 * w.grad
    w.grad.zero_()
    print(step, loss.item(), w.item())
```

```
0 9.0 1.5
1 2.25 2.25
2 0.5625 2.625
3 0.140625 2.8125
```

The loss is smallest at `w = 3`. Its derivative is 2 × (w − 3), so with a learning rate of 0.25 each
step moves `w` half of the way that is left, and the loss falls to a quarter each time.

## Setup

Two libraries, a way to quiet TensorFlow while it loads, and the stations.

- `numpy` builds the stations, and its least squares gives the exact answer to check the loop against
- `torch` is PyTorch
- `tensorflow` is TensorFlow, for the last worked example and one of the Common errors. `keras`
  loads with it and is not used here
- `warnings` catches the warning that one of the Common errors raises
- `contextlib` and `os` hide the messages TensorFlow prints while it loads

TensorFlow's C++ code prints several lines as it loads, each stamped with the time and a process
number, so no two runs would show the same output. Setting `TF_CPP_MIN_LOG_LEVEL` to 3 hides what
it logs once it has started, and `c_messages_hidden` sends the lines it prints while it starts to
`os.devnull`. The cell prints PyTorch's version without the part after `+`, which names the build.

`station_network_arrays` makes the 1,200 weather stations in Norway that the **scikit-learn, Deep
Dive** guide uses. The stations are made up, and a generator started from a fixed seed draws them,
so they are the same on every run. `features` holds each station's latitude, longitude, elevation
in meters and distance from the coast in kilometers, and `jan_mean` its January mean in degrees
Celsius. The cell splits the stations into 800 for training, 200 for validation and 200 for
testing, and `scaled` holds the features standardized with the training stations' means and
standard deviations. This notebook trains on the 800 training stations only.

In [1]:
import contextlib
import os
import warnings

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | 1200 stations


## Worked examples

### The parts of a single training step

The model predicts a station's January mean as `X @ w + b`, which multiplies each of its four
standardized features by a weight, adds them up and adds a bias. Every weight starts at 0. The cell
below takes one step and prints what each part produced.

In [2]:
X_train = torch.from_numpy(scaled[train_rows])
y_train = torch.from_numpy(jan_mean[train_rows])

w = torch.zeros(4, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

prediction = X_train @ w + b
loss = ((prediction - y_train) ** 2).mean()
print("prediction", tuple(prediction.shape), "| loss", round(loss.item(), 3))

loss.backward()
print("w.grad", w.grad.numpy().round(3), "| b.grad", round(b.grad.item(), 3))

with torch.no_grad():
    w -= 0.1 * w.grad
    b -= 0.1 * b.grad
print("w", w.detach().numpy().round(3), "| b", round(b.item(), 3))

w.grad.zero_()
b.grad.zero_()
print("w.grad", w.grad.numpy(), "| b.grad", b.grad.item())

prediction (800,) | loss 34.569
w.grad [3.97  3.891 3.056 2.924] | b.grad 10.215
w [-0.397 -0.389 -0.306 -0.292] | b -1.022
w.grad [0. 0. 0. 0.] | b.grad 0.0


Every prediction starts at 0, so the first loss, 34.569, is the mean square of the January means.
All four gradients are positive, so the step moves every weight below 0, by 0.1 times its gradient,
and latitude's gradient of 3.97 becomes a weight of −0.397.

Each step of the loop has the same five parts:

- **predict**: `X_train @ w + b` gives one number per station, 800 of them
- **measure**: the loss is the mean of the squared errors
- **differentiate**: `loss.backward()` fills `w.grad` and `b.grad`
- **step**: each parameter moves by 0.1 times its gradient, against its sign
- **zero**: the gradients go back to 0, because the next `backward()` would add to them

### The whole loop checked against the exact answer

The loop below repeats the same five parts and prints the loss after 0, 10, 100 and 1,000 steps.

In [3]:
w = torch.zeros(4, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
for step in range(1001):
    loss = ((X_train @ w + b - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
        b -= 0.1 * b.grad
    w.grad.zero_()
    b.grad.zero_()
    if step in (0, 10, 100, 1000):
        print(step, round(loss.item(), 3))

w_torch, b_torch = w.detach(), b.detach()
print("weights", w_torch.numpy().round(3), "| bias", round(b_torch.item(), 3))

0 34.569
10 1.347
100 1.006
1000 0.992
weights [-1.902 -0.137 -1.247 -1.055] | bias -5.108


A linear model's best weights can also be computed in one go. `np.linalg.lstsq` solves the least
squares problem directly, given the features with a column of ones added for the bias.

In [4]:
with_ones = np.column_stack([scaled[train_rows], np.ones(len(train_rows))])
exact = np.linalg.lstsq(with_ones, jan_mean[train_rows], rcond=None)[0]
print("weights", exact[:4].round(3), "| bias", round(float(exact[4]), 3))
print("loss", round(float(((with_ones @ exact - jan_mean[train_rows]) ** 2).mean()), 3))

weights [-1.903 -0.136 -1.247 -1.055] | bias -5.108
loss 0.992


The loop's weights for elevation and the coast, and its bias of −5.108, match the exact answer to the
third decimal place. Latitude and longitude still differ in the third decimal, −1.902 against
−1.903 and −0.137 against −0.136. The loss fell from 34.569 to 1.347 in the first ten steps and
to 1.006 by step 100, and the last 900 steps took it only to 0.992.

### Learning rates side by side

`train` runs the same loop for a given learning rate and number of steps, and returns the loss
before every step, so that runs can be compared. `X` defaults to the standardized training features.

In [5]:
def train(lr, steps, X=X_train):
    """Fit the four-feature linear model by gradient descent and return the loss before each step."""
    w = torch.zeros(4, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    losses = []
    for step in range(steps):
        loss = ((X @ w + b - y_train) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    return losses


print("rate    step 10  step 50  step 200")
for lr in (0.001, 0.01, 0.1, 0.5):
    losses = train(lr, 201)
    print(f"{lr:<7} {losses[10]:<8.3f} {losses[50]:<8.3f} {losses[200]:.3f}")

rate    step 10  step 50  step 200
0.001   33.073   27.764   14.821
0.01    22.309   4.826    1.040
0.1     1.347    1.019    0.996
0.5     2.962    1.159    0.992


At 0.001 the loss is still 14.821 after 200 steps, and at 0.01 it has reached 1.040. A rate of 0.1
is close to the exact answer's 0.992 by step 50. A larger rate is not always faster. At 0.5 the loss
trails 0.1's at steps 10 and 50, 2.962 against 1.347 and 1.159 against 1.019, and is below it only at
step 200, and a rate slightly above 0.5 fails altogether, as the Common errors show.

### Latitude and longitude move together

After 1,000 steps, the whole loop's weights for latitude and longitude still differed from the
exact answer in the third decimal place, while the other two had arrived. Across these made-up
stations, longitude rises with latitude, as it does in Norway.

In [6]:
print("correlation of latitude and longitude:",
      round(float(np.corrcoef(scaled[train_rows, 0], scaled[train_rows, 1])[0, 1]), 3))

w = torch.zeros(4, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
print("step  latitude  longitude  sum     difference")
for step in range(1, 1001):
    loss = ((X_train @ w + b - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
        b -= 0.1 * b.grad
    w.grad.zero_()
    b.grad.zero_()
    if step in (10, 50, 100, 200, 500, 1000):
        lat, lon = w[0].item(), w[1].item()
        print(f"{step:<5} {lat:<9.3f} {lon:<10.3f} {lat + lon:<7.3f} {lat - lon:.3f}")

correlation of latitude and longitude: 0.967
step  latitude  longitude  sum     difference
10    -1.059    -0.960     -2.019  -0.099
50    -1.260    -0.779     -2.039  -0.481
100   -1.439    -0.600     -2.039  -0.839
200   -1.662    -0.377     -2.039  -1.284
500   -1.869    -0.170     -2.039  -1.699
1000  -1.902    -0.137     -2.039  -1.765


The sum of the two weights reaches −2.039 by step 50 and stays there. Their difference is still
moving at step 1,000, from −0.099 at step 10 to −1.765. Because the two features rise together, with
a correlation of 0.967, raising one weight and lowering the other by the same amount barely changes
the predictions, so the loss is almost flat in that direction and the gradient along it is small.
Standardizing gives every feature the same spread and leaves the correlation as it was.

### Why the features are standardized

The size of each gradient follows the size of its feature. At the starting weights of 0, the
gradients for the standardized features and for the same features in their own units are:

In [7]:
X_raw = torch.from_numpy(features[train_rows])

for name, X in (("standardized", X_train), ("in their own units", X_raw)):
    w = torch.zeros(4, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    loss = ((X @ w + b - y_train) ** 2).mean()
    loss.backward()
    print(f"{name:<20}", w.grad.numpy().round(1))

standardized         [4.  3.9 3.1 2.9]
in their own units   [ 673.   198.2 3615.8  315. ]


A weight's gradient is 2 times the mean of the error times its feature, so a feature with large
values gets a large gradient. In meters, elevation's gradient is 3615.8, more than a thousand times
its standardized 3.1, and a learning rate of 0.1 would move its weight by about 362 in one step. The
Common errors below run that loop.

### Mini-batches and epochs

A mini-batch loop shuffles the training stations at the start of every epoch and steps once for
each batch of 32. `torch.randperm` gives a shuffled order of the row numbers, and a seeded
`torch.Generator` makes the orders the same on every run. With 800 stations, an epoch is 25 steps.
Each loss the loop computes is for one batch, so the cell also computes the loss on all 800 stations
at the end of a few epochs.

In [8]:
generator = torch.Generator().manual_seed(0)
w = torch.zeros(4, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
for epoch in range(1, 21):
    shuffled = torch.randperm(len(train_rows), generator=generator)
    for start in range(0, len(train_rows), 32):
        batch = shuffled[start:start + 32]
        loss = ((X_train[batch] @ w + b - y_train[batch]) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= 0.1 * w.grad
            b -= 0.1 * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if epoch in (1, 5, 10, 20):
        with torch.no_grad():
            all_loss = ((X_train @ w + b - y_train) ** 2).mean()
        print(f"epoch {epoch:<3} last batch {loss.item():.3f} | all 800 {all_loss.item():.3f}")

print("full batch after 20 steps:", round(train(0.1, 21)[20], 3))

epoch 1   last batch 1.306 | all 800 1.030
epoch 5   last batch 0.749 | all 800 1.010
epoch 10  last batch 1.060 | all 800 1.007
epoch 20  last batch 1.478 | all 800 0.999
full batch after 20 steps: 1.036


After one epoch, 25 steps on batches of 32 that read the data once, the loss on all 800 stations is
1.030, close to the 1.036 that twenty full-batch steps reach by reading the data twenty times. Which
of the two is lower depends on the order the shuffle drew. The last batch's loss jumps around,
1.306, 0.749, 1.060 and 1.478, because each batch is a different 32 stations, while the loss on all
800 falls steadily to 0.999. That is a little above the full-batch loop's 0.992, because every step
followed a noisy estimate of the gradient.

### The same loop in TensorFlow

In TensorFlow the parameters are `tf.Variable`s, the loss is computed inside a `GradientTape`, and
`assign_sub` makes the step. A tape gives fresh gradients each time, so there is nothing to zero.
`tf.linalg.matvec` multiplies each station's row of features by the weights, as `@` does above. A
TensorFlow tensor cannot be indexed with an array of row numbers as a PyTorch tensor can, and
`tf.gather(X, rows)` takes those rows instead.

In [9]:
X_train_tf = tf.constant(scaled[train_rows])
y_train_tf = tf.constant(jan_mean[train_rows])

w = tf.Variable(tf.zeros(4))
b = tf.Variable(0.0)
for step in range(1, 1001):
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean((tf.linalg.matvec(X_train_tf, w) + b - y_train_tf) ** 2)
    dw, db = tape.gradient(loss, [w, b])
    w.assign_sub(0.1 * dw)
    b.assign_sub(0.1 * db)

print("TensorFlow weights", w.numpy().round(3), "| bias", round(float(b), 3))
print("PyTorch weights   ", w_torch.numpy().round(3), "| bias", round(b_torch.item(), 3))

TensorFlow weights [-1.902 -0.137 -1.247 -1.055] | bias -5.108
PyTorch weights    [-1.902 -0.137 -1.247 -1.055] | bias -5.108


TensorFlow reaches the same weights and bias as PyTorch, to the third decimal place.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/05-gradient-descent-by-hand-solutions.ipynb).

**1.** Run `train` with a learning rate of 0.3 for 201 steps, and print the loss at steps 10, 50 and 200 beside the rows for 0.1 and 0.5 in the worked examples.

In [10]:
# your code here

**2.** Print the loss after 200 steps for learning rates of 0.50, 0.505, 0.51 and 0.52, formatted with `:.3g`, and find where the loop stops converging.

In [11]:
# your code here

**3.** Run the mini-batch loop with batches of 100 stations in place of 32, from `torch.Generator().manual_seed(0)`, and print the loss on all 800 stations after epochs 5, 10 and 20. How many steps is an epoch now?

In [12]:
# your code here

**4.** Train on the features in their own units, `X_raw`, with a learning rate of 1e-6 for 2,001 steps, and print the loss at steps 200 and 2,000 beside the standardized loop's.

In [13]:
# your code here

**5.** Write the TensorFlow loop with mini-batches of 32, shuffling the row numbers each epoch with `permutation` from one `np.random.default_rng(0)` made before the loop and taking each batch with `tf.gather`, and print the loss on all 800 stations after 20 epochs.

In [14]:
# your code here

## Common errors

### No error, and a loss of `nan` from a learning rate that is too large

In [15]:
losses = train(0.7, 201)
for step in (0, 1, 4, 9, 24, 49, 74, 99, 149, 200):
    print(step, f"{losses[step]:.3g}")

0 34.6
1 19.2
4 330
9 9.22e+04
24 2.05e+12
49 3.59e+24
74 inf
99 inf
149 inf
200 nan


The loss falls once, to 19.2, and then grows at every step: 330 at step 4, 92,200 at step 9 and
3.59e+24 at step 49. By step 74 it is larger than a 32-bit float can hold, which prints as `inf`,
and by step 200 it is `nan`, short for "not a number", the result of arithmetic such as infinity
minus infinity, with no error raised at any step.

In [16]:
for lr in (0.5, 0.505, 0.51):
    print(lr, f"{train(lr, 201)[200]:.3g}")

0.5 0.992
0.505 1.04
0.51 131


Between 0.5 and 0.51 the loop goes from converging to diverging: 0.5 reaches 0.992, 0.505 is at
1.04 after 200 steps, and 0.51 is at 131. Where that edge falls depends on the data, so lower the
learning rate until the loss falls steadily, and read the loss after training instead of taking the
absence of an error as success.

### No error, and features in their own units that diverge at 0.1 and crawl at 1e-6

In [17]:
for name, X, lr in (("standardized, 0.1", X_train, 0.1),
                    ("own units, 0.1", X_raw, 0.1),
                    ("own units, 1e-6", X_raw, 1e-6)):
    losses = train(lr, 201, X)
    print(f"{name:<18}", [f"{losses[step]:.3g}" for step in (0, 10, 50, 200)])

standardized, 0.1  ['34.6', '1.35', '1.02', '0.996']
own units, 0.1     ['34.6', 'nan', 'nan', 'nan']
own units, 1e-6    ['34.6', '8.07', '7.31', '5.7']


At 0.1, the rate that brings the standardized features to 0.996, the features in their own units
send the loss to `nan` within 10 steps. At 1e-6, slow enough for elevation, the loss is still 5.7
after 200 steps, because latitude, whose gradient was about a fifth of elevation's, now takes tiny steps
too. Standardize the features with the training stations' means and standard deviations, as Setup
does, and one learning rate suits all four.

### `TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'`

This loop writes the step as `w = w - 0.1 * w.grad`, outside `torch.no_grad()`. The cell records any
`UserWarning` so that it can be printed in the next cell.

In [18]:
w = torch.zeros(4, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    for step in range(3):
        loss = ((X_train @ w + b - y_train) ** 2).mean()
        loss.backward()
        w = w - 0.1 * w.grad
        b = b - 0.1 * b.grad

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

In [19]:
print("step", step, "| w is a leaf:", w.is_leaf)
for warning in caught[:1]:
    print(f"{warning.category.__name__}: {str(warning.message).split('. ')[0]}.")

step 1 | w is a leaf: False


On the first pass, `w - 0.1 * w.grad` made a new tensor and the name `w` moved to it. That tensor
was computed from the weights, so it is not a leaf, and `backward()` fills `.grad` only on leaves,
the tensors made directly with `requires_grad=True`. On the second pass `w.grad` was `None`, PyTorch
warned, and `0.1 * None` raised the `TypeError`. The warning names its source file inside PyTorch, so
the cell prints only its first sentence. The fix is the worked examples' form, a step inside
`torch.no_grad()` written with `-=`, which changes the leaf itself.

TensorFlow raises the same error for the same mistake.

In [20]:
w = tf.Variable(tf.zeros(4))
b = tf.Variable(0.0)
for step in range(3):
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean((tf.linalg.matvec(X_train_tf, w) + b - y_train_tf) ** 2)
    dw, db = tape.gradient(loss, [w, b])
    w = w - 0.1 * dw
    b = b - 0.1 * db

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

`w - 0.1 * dw` is a plain tensor, and a tape watches a `tf.Variable` by itself and a plain tensor
only when told to with `tape.watch`, so on the second pass `tape.gradient` returned `None` for it.
`w.assign_sub(0.1 * dw)` changes the variable in place, and it stays a variable.

## Recap

- You can write the five parts of a training step by hand, in PyTorch and in TensorFlow.
- You can check a loop's answer against least squares, and see why two correlated weights settle
  slowly.
- You can tell a learning rate that creeps from one that diverges, and why standardizing the
  features lets one rate suit them all.
- You can train on mini-batches, shuffled each epoch from a seeded generator.

## What is next

The **Layers and Models** notebook puts the weights inside `nn.Module` and `keras.Model`, which hold
them, count them and build a network out of layers.

---

&#8592; **Previous:** [Automatic Differentiation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/04-automatic-differentiation.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Layers and Models](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/06-layers-and-models.ipynb) &#8594;
